# Chrome Dino CNN Training
Cleaned notebook based on the original bootcamp training code.


In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


In [ ]:
class ObstacleDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []
        root = Path(root_dir)
        for label_name, label in [('no', 0), ('yes', 1)]:
            folder = root / label_name
            if not folder.exists():
                continue
            for path in folder.iterdir():
                if path.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp'}:
                    self.samples.append((str(path), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.zeros((64, 256), dtype=np.uint8)
        else:
            img = cv2.resize(img, (256, 64))
        img = torch.tensor(img / 255.0, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(label, dtype=torch.float32)
        return img, label

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.fc = nn.Linear(64 * 16 * 64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = x.view(x.size(0), -1)
        return self.sigmoid(self.fc(x))


In [ ]:
# Set these paths to the extracted training/test folders.
TRAIN_ROOT = 'train_data/train'
TEST_ROOT = 'test_data/test'

train_dataset = ObstacleDataset(TRAIN_ROOT)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

model = CNN().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{epochs}] Loss: {total_loss:.4f}')


In [ ]:
test_dataset = ObstacleDataset(TEST_ROOT)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        labels = labels.to(device).unsqueeze(1)
        outputs = model(imgs)
        predicted = (outputs > 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

if total:
    print(f'Test Accuracy: {100 * correct / total:.2f}%')

torch.save(model.state_dict(), 'dino_cnn_model.pth')
